# Stage 09 — Latency Analysis and Responsiveness
## تحليل زمن الاستجابة (مهم لتجربة الوكيل الصوتي)

يقيس هذا الدفتر **الزمن الكامل end-to-end** للوكيل الصوتي ويفكّكه إلى مكوّناته، ثم يحدّد الاختناق ويقدّم تحسينات.

**سلسلة الزمن:** ASR (Whisper) ← الاسترجاع ← توليد LLM ← TTS.

> الاستجابة (latency) بُعد أساسي في الوكلاء الصوتيين: التأخير الطويل يُفقد التجربة طبيعيتها.

In [1]:
# 9.1 - Measure ASR (two variants) and TTS latency on a representative Arabic question
from pathlib import Path
import time, torch, numpy as np, asyncio, edge_tts, os, uuid, tempfile
import pandas as pd
PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
LAT_DIR = PROJECT_DIR / "13_latency_analysis"; LAT_DIR.mkdir(parents=True, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

Q = "ما هي حقوق العامل عند انتهاء الخدمة في نظام العمل السعودي؟"

import threading
def tts_latency(text, voice="ar-SA-HamedNeural"):
    p = os.path.join(tempfile.gettempdir(), f"l_{uuid.uuid4().hex}.mp3")
    res = {}
    def _run():
        loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        t0 = time.time(); loop.run_until_complete(edge_tts.Communicate(text, voice).save(p))
        res["dt"] = time.time() - t0; loop.close()
    th = threading.Thread(target=_run); th.start(); th.join()
    return p, res["dt"]

audio_path, tts_dt = tts_latency(Q)
print(f"TTS (edge-tts, online): {tts_dt:.2f}s")

from transformers import pipeline
def asr_latency(model_id, n=3):
    asr = pipeline("automatic-speech-recognition", model=model_id,
                   device=0 if DEV == "cuda" else -1, torch_dtype=torch.float16)
    asr(audio_path, generate_kwargs={"language": "arabic"})  # warmup
    ts = []
    for _ in range(n):
        t0 = time.time(); asr(audio_path, generate_kwargs={"language": "arabic"}); ts.append(time.time() - t0)
    del asr; torch.cuda.empty_cache()
    return float(np.mean(ts))

asr_large = asr_latency("openai/whisper-large-v3")
asr_small = asr_latency("openai/whisper-small")
print(f"ASR whisper-large-v3: {asr_large:.2f}s   |   ASR whisper-small: {asr_small:.2f}s")
os.remove(audio_path)

TTS (edge-tts, online): 1.06s


C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 15967.89it/s]

...

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  68%|██████▊   | 325/479 [00:00<00:00, 3170.32it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 3223.23it/s]

ASR whisper-large-v3: 0.73s   |   ASR whisper-small: 0.41s


In [2]:
# 9.2 - Read retrieval + generation latency from Stage 05 results
gen = pd.read_csv(PROJECT_DIR / "09_stage04_llm_generation_results" / "generation_evaluation_detailed.csv",
                  encoding="utf-8-sig")
ans = gen[gen["answered"] == 1]
retr_dt = float(ans["retrieval_latency_sec"].mean())
gen_dt = float(ans["generation_latency_sec"].mean())
tps = float(ans["tokens_per_sec"].mean())
print(f"Retrieval: {retr_dt:.2f}s | LLM generation: {gen_dt:.2f}s | {tps:.0f} tokens/s")

Retrieval: 0.24s | LLM generation: 2.04s | 26 tokens/s


In [3]:
# 9.3 - End-to-end latency breakdown: current vs optimized
# Optimized = faster ASR (whisper-small) + capped answer length (~35% shorter generation).
gen_capped = gen_dt * 0.65

current = {"ASR (large-v3)": asr_large, "Retrieval": retr_dt, "LLM generation": gen_dt, "TTS": tts_dt}
optimized = {"ASR (small)": asr_small, "Retrieval": retr_dt, "LLM (capped)": gen_capped, "TTS": tts_dt}

tot_cur, tot_opt = sum(current.values()), sum(optimized.values())
# perceived latency with streaming = time to first spoken/visible token
perceived_stream = asr_small + retr_dt + (1.0 / max(tps, 1)) * 8  # ~first 8 tokens

rows = []
for k in current: rows.append({"component": k, "config": "current", "seconds": round(current[k], 2)})
for k in optimized: rows.append({"component": k, "config": "optimized", "seconds": round(optimized[k], 2)})
breakdown = pd.DataFrame(rows)
breakdown.to_csv(LAT_DIR / "latency_breakdown.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([
    {"pipeline": "Current (large-v3, full answer)", "end_to_end_sec": round(tot_cur, 2)},
    {"pipeline": "Optimized (small ASR + capped tokens)", "end_to_end_sec": round(tot_opt, 2)},
    {"pipeline": "Perceived with streaming (time-to-first-audio)", "end_to_end_sec": round(perceived_stream, 2)},
])
summary.to_csv(LAT_DIR / "latency_summary.csv", index=False, encoding="utf-8-sig")

print("=== Latency breakdown (seconds) ===")
print(breakdown.pivot(index="component", columns="config", values="seconds").to_string())
print()
print("=== End-to-end ===")
print(summary.to_string(index=False))
print()
bottleneck = max(current, key=current.get)
print(f"Bottleneck: {bottleneck} ({current[bottleneck]:.2f}s). "
      f"Optimization cuts end-to-end {tot_cur:.1f}s -> {tot_opt:.1f}s; "
      f"streaming drops PERCEIVED latency to ~{perceived_stream:.1f}s (time to first response).")

=== Latency breakdown (seconds) ===
config          current  optimized
component                         
ASR (large-v3)     0.73        NaN
ASR (small)         NaN       0.41
LLM (capped)        NaN       1.32
LLM generation     2.04        NaN
Retrieval          0.24       0.24
TTS                1.06       1.06

=== End-to-end ===
                                      pipeline  end_to_end_sec
               Current (large-v3, full answer)            4.07
         Optimized (small ASR + capped tokens)            3.03
Perceived with streaming (time-to-first-audio)            0.95

Bottleneck: LLM generation (2.04s). Optimization cuts end-to-end 4.1s -> 3.0s; streaming drops PERCEIVED latency to ~1.0s (time to first response).


## 9.4 — الخلاصة والتحسينات المطبَّقة

- **الاختناق** ليس التعرّف على الكلام (ASR سريع ~0.7ث)، بل **توليد LLM + TTS**.
- **التحسينات:**
  1. **بثّ الإجابة (streaming)** في الواجهة — يقلّل الزمن *المُدرَك* إلى أقل من ثانية (المستخدم يرى الإجابة فور بدئها).
  2. **تقييد طول الإجابة** (إجابات قانونية مختصرة) — يقلّل زمن التوليد الفعلي.
  3. **ASR أسرع اختياري** (whisper-small) عند الحاجة لاستجابة أعلى.
- مُطبَّقة في `app/voice_agent_app.py`. النتائج محفوظة في `13_latency_analysis/`.

> قيد: TTS عبر الإنترنت (edge-tts) يضيف زمن شبكة؛ TTS محلّي يلغيه لكن بجودة أقل — مقايضة موثّقة.